# FACE TRACE MACHINE LEARNING MODEL: Speech Recognition → NLP → Sketch Generation
### Large-Scale Dataset (100,000+ samples) | 10-15 Algorithms | Sequential Pipeline
---
**Pipeline Flow:**
```
Audio Features → [Speech Recognition] → Transcript
     ↓
Transcript → [Natural Language Processing] → Intent/Entities
     ↓
Intent → [Sketch Generation] → Generated Sketch Image
```

## Section 0: Install & Import All Dependencies

In [2]:
# Install required packages
import subprocess, sys
packages = [
    'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn',
    'scipy', 'librosa', 'Pillow', 'tqdm', 'xgboost',
    'lightgbm', 'nltk', 'wordcloud', 'imbalanced-learn'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages installed successfully!')

✅ All packages installed successfully!


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm import tqdm
import warnings, os, time, random, string
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, ExtraTreesClassifier,
                              BaggingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.multiclass import OneVsRestClassifier

# XGBoost & LightGBM
import xgboost as xgb
import lightgbm as lgb

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Image
from PIL import Image, ImageDraw
import scipy.ndimage as ndimage

np.random.seed(42)
random.seed(42)

print('All imports successful!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__} | Sklearn: ready')

---
# PART 1: SPEECH RECOGNITION
## Dataset Generation + 10-15 Algorithms

### 1.1 Generate Large Speech Recognition Dataset (100,000 samples)

In [ ]:
print('Generating Speech Recognition Dataset (100,000 samples)...')

N_SPEECH = 100_000

# 10 phoneme classes / spoken word categories
SPEECH_CLASSES = [
    'circle', 'square', 'triangle', 'star', 'house',
    'tree', 'car', 'cat', 'dog', 'flower'
]

def generate_audio_features(word_class, n_samples):
    """Simulate MFCC-like audio features per word class."""
    # Each class has distinct spectral characteristics
    class_idx = SPEECH_CLASSES.index(word_class)
    base_freq = 100 + class_idx * 80          # fundamental frequency
    base_energy = 0.3 + class_idx * 0.06      # energy envelope
    base_duration = 0.2 + class_idx * 0.03    # spoken duration (normalized)
    base_zcr = 0.05 + class_idx * 0.02        # zero crossing rate

    features = {
        # MFCC coefficients (13 coefficients)
        **{f'mfcc_{i}': np.random.normal(base_freq + i*10, 15, n_samples) for i in range(13)},
        # Delta MFCCs
        **{f'delta_mfcc_{i}': np.random.normal(i*2, 5, n_samples) for i in range(6)},
        # Spectral features
        'spectral_centroid':  np.random.normal(base_freq * 3.5, 50, n_samples),
        'spectral_rolloff':   np.random.normal(base_freq * 6, 80, n_samples),
        'spectral_bandwidth': np.random.normal(base_freq * 2, 40, n_samples),
        'spectral_contrast':  np.random.normal(20 + class_idx*3, 5, n_samples),
        # Temporal features
        'zero_crossing_rate': np.abs(np.random.normal(base_zcr, 0.01, n_samples)),
        'rms_energy':         np.abs(np.random.normal(base_energy, 0.05, n_samples)),
        'duration':           np.abs(np.random.normal(base_duration, 0.05, n_samples)),
        'tempo':              np.random.normal(120 + class_idx*5, 10, n_samples),
        # Pitch
        'pitch_mean':         np.random.normal(base_freq * 1.5, 20, n_samples),
        'pitch_std':          np.abs(np.random.normal(15 + class_idx, 5, n_samples)),
        'pitch_max':          np.random.normal(base_freq * 2, 30, n_samples),
        # Formants
        'f1': np.random.normal(400 + class_idx*50, 40, n_samples),
        'f2': np.random.normal(1200 + class_idx*80, 60, n_samples),
        'f3': np.random.normal(2500 + class_idx*40, 80, n_samples),
        # Noise/quality
        'snr': np.random.normal(20 + class_idx*2, 3, n_samples),
        'label': [word_class] * n_samples
    }
    return pd.DataFrame(features)

# Generate balanced dataset
samples_per_class = N_SPEECH // len(SPEECH_CLASSES)
speech_dfs = []
for wc in tqdm(SPEECH_CLASSES, desc='Generating speech data'):
    speech_dfs.append(generate_audio_features(wc, samples_per_class))

speech_df = pd.concat(speech_dfs, ignore_index=True)
speech_df = speech_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n Speech Dataset Shape: {speech_df.shape}')
print(f'Classes: {SPEECH_CLASSES}')
print(f'\nClass Distribution:')
print(speech_df['label'].value_counts())

In [ ]:
# Visualize Speech Dataset
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(' Speech Recognition Dataset Analysis', fontsize=16, fontweight='bold')

# Class distribution
speech_df['label'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Class Distribution (100K samples)')
axes[0,0].set_xlabel('Word Class'); axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=45)

# MFCC distribution per class
for cls in SPEECH_CLASSES[:5]:
    subset = speech_df[speech_df['label']==cls]
    axes[0,1].hist(subset['mfcc_0'], bins=30, alpha=0.4, label=cls)
axes[0,1].set_title('MFCC_0 Distribution by Class')
axes[0,1].legend(fontsize=7); axes[0,1].set_xlabel('MFCC_0 Value')

# Spectral centroid vs energy
sample = speech_df.sample(2000)
le_temp = LabelEncoder()
colors = le_temp.fit_transform(sample['label'])
sc = axes[0,2].scatter(sample['spectral_centroid'], sample['rms_energy'], 
                        c=colors, cmap='tab10', alpha=0.5, s=10)
axes[0,2].set_title('Spectral Centroid vs RMS Energy')
axes[0,2].set_xlabel('Spectral Centroid'); axes[0,2].set_ylabel('RMS Energy')

# Pitch distribution
speech_df.groupby('label')['pitch_mean'].mean().plot(kind='bar', ax=axes[1,0], color='coral', edgecolor='black')
axes[1,0].set_title('Mean Pitch by Word Class')
axes[1,0].set_xlabel('Word Class'); axes[1,0].tick_params(axis='x', rotation=45)

# Correlation heatmap (subset of features)
feat_cols = ['mfcc_0','mfcc_1','spectral_centroid','rms_energy','zero_crossing_rate','pitch_mean','f1','f2']
corr = speech_df[feat_cols].corr()
sns.heatmap(corr, ax=axes[1,1], cmap='coolwarm', annot=True, fmt='.2f', fontsize=7)
axes[1,1].set_title('Feature Correlation Heatmap')

# ZCR boxplot
speech_df.boxplot(column='zero_crossing_rate', by='label', ax=axes[1,2])
axes[1,2].set_title('Zero Crossing Rate by Class')
axes[1,2].set_xlabel('Word Class'); axes[1,2].tick_params(axis='x', rotation=45)
plt.sca(axes[1,2]); plt.title('ZCR by Class')

plt.tight_layout()
plt.savefig('speech_dataset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Speech dataset visualization complete!')

### 1.2 Preprocessing & Feature Engineering

In [ ]:
print('🔧 Preprocessing Speech Data...')

# Encode labels
le_speech = LabelEncoder()
speech_df['label_enc'] = le_speech.fit_transform(speech_df['label'])

# Feature columns
speech_feature_cols = [c for c in speech_df.columns if c not in ['label', 'label_enc']]

X_speech = speech_df[speech_feature_cols].values
y_speech = speech_df['label_enc'].values

# Train/Test split
X_sp_train, X_sp_test, y_sp_train, y_sp_test = train_test_split(
    X_speech, y_speech, test_size=0.2, random_state=42, stratify=y_speech
)

# Scale features
scaler_speech = StandardScaler()
X_sp_train_sc = scaler_speech.fit_transform(X_sp_train)
X_sp_test_sc  = scaler_speech.transform(X_sp_test)

print(f'Train: {X_sp_train_sc.shape} | Test: {X_sp_test_sc.shape}')
print(f'Features: {len(speech_feature_cols)}')
print(f'Classes: {le_speech.classes_}')

### 1.3 Apply 12 Algorithms for Speech Recognition

In [ ]:
# Define 12 algorithms
speech_algorithms = {
    '1. Random Forest':          RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42),
    '2. XGBoost':                xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                                    use_label_encoder=False, eval_metric='mlogloss',
                                                    n_jobs=-1, random_state=42),
    '3. LightGBM':               lgb.LGBMClassifier(n_estimators=200, max_depth=10, learning_rate=0.1,
                                                     n_jobs=-1, random_state=42, verbose=-1),
    '4. Gradient Boosting':      GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
    '5. Extra Trees':            ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    '6. MLP Neural Network':     MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=300,
                                               random_state=42, early_stopping=True),
    '7. SVM (Linear)':           LinearSVC(max_iter=2000, random_state=42),
    '8. Logistic Regression':    LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, random_state=42),
    '9. KNN':                    KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    '10. AdaBoost':              AdaBoostClassifier(n_estimators=150, random_state=42),
    '11. Decision Tree':         DecisionTreeClassifier(max_depth=20, random_state=42),
    '12. SGD Classifier':        SGDClassifier(max_iter=1000, random_state=42, n_jobs=-1),
}

speech_results = {}

print('🚀 Training 12 Speech Recognition Algorithms...\n')
print(f'{"Algorithm":<30} {"Accuracy":>10} {"F1-Score":>10} {"Time(s)":>10}')
print('-' * 65)

best_speech_model = None
best_speech_acc = 0
best_speech_name = ''

for name, model in speech_algorithms.items():
    start = time.time()
    
    # Use scaled data for distance/gradient methods, raw for tree methods
    use_scaled = any(k in name for k in ['SVM','Logistic','MLP','KNN','SGD'])
    Xtr = X_sp_train_sc if use_scaled else X_sp_train
    Xte = X_sp_test_sc  if use_scaled else X_sp_test
    
    model.fit(Xtr, y_sp_train)
    y_pred = model.predict(Xte)
    
    acc = accuracy_score(y_sp_test, y_pred)
    f1  = f1_score(y_sp_test, y_pred, average='weighted')
    elapsed = time.time() - start
    
    speech_results[name] = {'accuracy': acc, 'f1': f1, 'time': elapsed, 'model': model}
    print(f'{name:<30} {acc:>10.4f} {f1:>10.4f} {elapsed:>10.2f}s')
    
    if acc > best_speech_acc:
        best_speech_acc = acc
        best_speech_model = model
        best_speech_name = name

print('-' * 65)
print(f'\n🏆 Best Speech Model: {best_speech_name} | Accuracy: {best_speech_acc:.4f}')

In [ ]:
# Ensemble Voting (bonus algorithm #13)
print('\n Training Ensemble Voting Classifier (Algorithm #13)...')

ensemble_speech = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)),
        ('xgb', xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss', n_jobs=-1, random_state=42)),
        ('lgb', lgb.LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=42, verbose=-1)),
        ('et',  ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=42))
    ], voting='hard', n_jobs=-1
)
start = time.time()
ensemble_speech.fit(X_sp_train, y_sp_train)
y_pred_ens = ensemble_speech.predict(X_sp_test)
ens_acc = accuracy_score(y_sp_test, y_pred_ens)
ens_f1  = f1_score(y_sp_test, y_pred_ens, average='weighted')
elapsed = time.time() - start
speech_results['13. Ensemble Voting'] = {'accuracy': ens_acc, 'f1': ens_f1, 'time': elapsed}
print(f'13. Ensemble Voting               {ens_acc:>10.4f} {ens_f1:>10.4f} {elapsed:>10.2f}s')

if ens_acc > best_speech_acc:
    best_speech_acc = ens_acc
    best_speech_model = ensemble_speech
    best_speech_name = '13. Ensemble Voting'

print(f'\n Final Best Speech Model: {best_speech_name} | Accuracy: {best_speech_acc:.4f}')

In [ ]:
# Visualize Speech Results
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('🎙️ Speech Recognition - Algorithm Comparison', fontsize=14, fontweight='bold')

names_sp = list(speech_results.keys())
accs_sp  = [speech_results[n]['accuracy'] for n in names_sp]
f1s_sp   = [speech_results[n]['f1'] for n in names_sp]
short_names = [n.split('. ')[1] for n in names_sp]

# Accuracy bar
colors_bar = ['gold' if a == max(accs_sp) else 'steelblue' for a in accs_sp]
axes[0].barh(short_names, accs_sp, color=colors_bar, edgecolor='black')
axes[0].axvline(0.99, color='red', linestyle='--', label='99% target')
axes[0].set_xlim(0.9, 1.01)
axes[0].set_title('Accuracy by Algorithm')
axes[0].set_xlabel('Accuracy'); axes[0].legend()
for i, v in enumerate(accs_sp):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# F1 bar
colors_f1 = ['gold' if f == max(f1s_sp) else 'coral' for f in f1s_sp]
axes[1].barh(short_names, f1s_sp, color=colors_f1, edgecolor='black')
axes[1].set_xlim(0.9, 1.01)
axes[1].set_title('F1-Score by Algorithm')
axes[1].set_xlabel('F1-Score')
for i, v in enumerate(f1s_sp):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# Confusion matrix for best model
use_scaled_best = any(k in best_speech_name for k in ['SVM','Logistic','MLP','KNN','SGD'])
Xte_best = X_sp_test_sc if use_scaled_best else X_sp_test
y_pred_best = best_speech_model.predict(Xte_best)
cm = confusion_matrix(y_sp_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=le_speech.classes_, yticklabels=le_speech.classes_)
axes[2].set_title(f'Confusion Matrix\n({best_speech_name})')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('speech_results.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n📊 Speech Recognition Summary:')
print(f'   Best Model: {best_speech_name}')
print(f'   Accuracy:   {best_speech_acc:.4f} ({best_speech_acc*100:.2f}%)')
print(f'   F1-Score:   {max(f1s_sp):.4f}')

---
# PART 2: NATURAL LANGUAGE PROCESSING (NLP)
## Dataset Generation + 12 Algorithms

### 2.1 Generate Large NLP Dataset (100,000 samples)

In [ ]:
print('Generating NLP Dataset (100,000 samples)...')

N_NLP = 100_000

# Intent categories (NLP classification task)
NLP_INTENTS = [
    'draw_circle', 'draw_square', 'draw_triangle', 'draw_star', 'draw_house',
    'draw_tree', 'draw_car', 'draw_cat', 'draw_dog', 'draw_flower'
]

# Template sentences per intent
TEMPLATES = {
    'draw_circle':   [
        'draw a circle', 'make a circle', 'sketch a round shape', 'create a circular object',
        'please draw me a circle', 'I want a circle', 'can you draw a round thing',
        'sketch something circular', 'generate a circle shape', 'make a round sketch'
    ],
    'draw_square':   [
        'draw a square', 'make a box shape', 'sketch a rectangle', 'create a square shape',
        'please draw me a square', 'I want a box', 'can you draw a square thing',
        'sketch something square', 'generate a box shape', 'make a four-sided sketch'
    ],
    'draw_triangle': [
        'draw a triangle', 'make a triangular shape', 'sketch a three-sided figure',
        'create a triangle', 'please draw me a triangle', 'I want a pyramidal shape',
        'can you draw a three-pointed shape', 'sketch something triangular',
        'generate a triangle', 'make a pointed sketch'
    ],
    'draw_star':     [
        'draw a star', 'make a star shape', 'sketch a five-pointed star',
        'create a star design', 'please draw me a star', 'I want a shining star',
        'can you draw a star', 'sketch something like a star', 'generate a star',
        'make a star pattern'
    ],
    'draw_house':    [
        'draw a house', 'make a home sketch', 'sketch a building with a roof',
        'create a house shape', 'please draw me a house', 'I want a home drawing',
        'can you draw a house', 'sketch something like a house', 'generate a house',
        'make a home illustration'
    ],
    'draw_tree':     [
        'draw a tree', 'make a tree sketch', 'sketch a plant with branches',
        'create a tree design', 'please draw me a tree', 'I want a tree drawing',
        'can you draw a tree', 'sketch something like a tree', 'generate a tree',
        'make a tree illustration'
    ],
    'draw_car':      [
        'draw a car', 'make a vehicle sketch', 'sketch an automobile',
        'create a car design', 'please draw me a car', 'I want a car drawing',
        'can you draw a vehicle', 'sketch something like a car', 'generate a car',
        'make a car illustration'
    ],
    'draw_cat':      [
        'draw a cat', 'make a feline sketch', 'sketch a kitten',
        'create a cat drawing', 'please draw me a cat', 'I want a cat illustration',
        'can you draw a cat', 'sketch something like a cat', 'generate a cat',
        'make a cat picture'
    ],
    'draw_dog':      [
        'draw a dog', 'make a dog sketch', 'sketch a puppy',
        'create a dog drawing', 'please draw me a dog', 'I want a dog illustration',
        'can you draw a dog', 'sketch something like a dog', 'generate a dog',
        'make a dog picture'
    ],
    'draw_flower':   [
        'draw a flower', 'make a floral sketch', 'sketch a bloom',
        'create a flower design', 'please draw me a flower', 'I want a flower drawing',
        'can you draw a flower', 'sketch something like a flower', 'generate a flower',
        'make a floral illustration'
    ],
}

# Noise words to vary sentences
PREFIXES  = ['', 'please ', 'can you ', 'would you ', 'I need you to ', 'kindly ', '']
SUFFIXES  = ['', ' for me', ' now', ' quickly', ' on the canvas', ' please', '']
ADJECTIVES= ['', 'beautiful ', 'simple ', 'large ', 'small ', 'colorful ', 'detailed ', '']

def generate_sentence(intent):
    base = random.choice(TEMPLATES[intent])
    # Insert adjective after 'a ' or 'an '
    adj = random.choice(ADJECTIVES)
    if adj:
        base = base.replace('a ', f'a {adj}', 1).replace('an ', f'an {adj}', 1)
    prefix = random.choice(PREFIXES)
    suffix = random.choice(SUFFIXES)
    sentence = f'{prefix}{base}{suffix}'.strip()
    # Random capitalization
    if random.random() > 0.5:
        sentence = sentence.capitalize()
    return sentence

# Generate dataset
nlp_data = []
samples_per_intent = N_NLP // len(NLP_INTENTS)

for intent in tqdm(NLP_INTENTS, desc='Generating NLP data'):
    for _ in range(samples_per_intent):
        sentence = generate_sentence(intent)
        nlp_data.append({'text': sentence, 'intent': intent})

nlp_df = pd.DataFrame(nlp_data).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n NLP Dataset Shape: {nlp_df.shape}')
print(f'\nSample sentences:')
print(nlp_df.head(10).to_string())

### 2.2 NLP Preprocessing & Feature Engineering

In [ ]:
print('Preprocessing NLP Data...')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower().strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

print('Preprocessing text (lemmatization, tokenization)...')
nlp_df['processed_text'] = [preprocess_text(t) for t in tqdm(nlp_df['text'], desc='Processing')]

# Additional handcrafted features
nlp_df['text_length']  = nlp_df['text'].apply(len)
nlp_df['word_count']   = nlp_df['text'].apply(lambda x: len(x.split()))
nlp_df['has_please']   = nlp_df['text'].str.lower().str.contains('please').astype(int)
nlp_df['has_draw']     = nlp_df['text'].str.lower().str.contains('draw').astype(int)
nlp_df['has_sketch']   = nlp_df['text'].str.lower().str.contains('sketch').astype(int)
nlp_df['has_make']     = nlp_df['text'].str.lower().str.contains('make').astype(int)
nlp_df['has_create']   = nlp_df['text'].str.lower().str.contains('create').astype(int)

# Encode labels
le_nlp = LabelEncoder()
nlp_df['label_enc'] = le_nlp.fit_transform(nlp_df['intent'])

# TF-IDF Features
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(nlp_df['processed_text'])
print(f'TF-IDF matrix shape: {X_tfidf.shape}')

# Count vectorizer
count_vec = CountVectorizer(ngram_range=(1,2), max_features=3000)
X_count = count_vec.fit_transform(nlp_df['processed_text'])

y_nlp = nlp_df['label_enc'].values

# Split
X_nlp_train, X_nlp_test, y_nlp_train, y_nlp_test = train_test_split(
    X_tfidf, y_nlp, test_size=0.2, random_state=42, stratify=y_nlp
)

print(f'Train: {X_nlp_train.shape} | Test: {X_nlp_test.shape}')
print('NLP Preprocessing complete!')

In [ ]:
# NLP Dataset Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('NLP Dataset Analysis', fontsize=14, fontweight='bold')

nlp_df['intent'].value_counts().plot(kind='bar', ax=axes[0], color='mediumorchid', edgecolor='black')
axes[0].set_title('Intent Distribution'); axes[0].tick_params(axis='x', rotation=45)

nlp_df['word_count'].hist(bins=20, ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Word Count Distribution'); axes[1].set_xlabel('Words per sentence')

nlp_df.groupby('intent')['text_length'].mean().plot(kind='bar', ax=axes[2], color='salmon', edgecolor='black')
axes[2].set_title('Avg Text Length per Intent'); axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('nlp_dataset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.3 Apply 12 Algorithms for NLP Intent Classification

In [ ]:
import scipy.sparse as sp

# Convert for tree-based models (need dense or sparse)
nlp_algorithms = {
    '1. Logistic Regression':     LogisticRegression(max_iter=1000, C=5.0, n_jobs=-1, random_state=42),
    '2. Linear SVM':              LinearSVC(max_iter=3000, C=1.0, random_state=42),
    '3. Multinomial NB':          MultinomialNB(alpha=0.1),
    '4. Bernoulli NB':            BernoulliNB(alpha=0.1),
    '5. SGD Classifier':          SGDClassifier(loss='modified_huber', max_iter=1000, random_state=42, n_jobs=-1),
    '6. Random Forest':           RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    '7. Extra Trees':             ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    '8. XGBoost':                 xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss', n_jobs=-1, random_state=42),
    '9. LightGBM':                lgb.LGBMClassifier(n_estimators=200, n_jobs=-1, random_state=42, verbose=-1),
    '10. Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    '11. MLP Neural Network':     MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300,
                                                random_state=42, early_stopping=True),
    '12. KNN':                    KNeighborsClassifier(n_neighbors=5, n_jobs=-1, metric='cosine'),
}

nlp_results = {}

print('Training 12 NLP Algorithms...\n')
print(f'{"Algorithm":<30} {"Accuracy":>10} {"F1-Score":>10} {"Time(s)":>10}')
print('-' * 65)

best_nlp_model = None
best_nlp_acc   = 0
best_nlp_name  = ''
best_nlp_preds = None

for name, model in nlp_algorithms.items():
    start = time.time()
    
    # Tree models need dense arrays from sparse
    dense_needed = any(k in name for k in ['Random Forest','Extra Trees','XGBoost','LightGBM',
                                            'Gradient Boosting','MLP','KNN'])
    if dense_needed:
        Xtr = X_nlp_train.toarray()
        Xte = X_nlp_test.toarray()
    else:
        Xtr = X_nlp_train
        Xte = X_nlp_test
    
    model.fit(Xtr, y_nlp_train)
    y_pred = model.predict(Xte)
    
    acc = accuracy_score(y_nlp_test, y_pred)
    f1  = f1_score(y_nlp_test, y_pred, average='weighted')
    elapsed = time.time() - start
    
    nlp_results[name] = {'accuracy': acc, 'f1': f1, 'time': elapsed}
    print(f'{name:<30} {acc:>10.4f} {f1:>10.4f} {elapsed:>10.2f}s')
    
    if acc > best_nlp_acc:
        best_nlp_acc   = acc
        best_nlp_model = model
        best_nlp_name  = name
        best_nlp_preds = y_pred

print('-' * 65)
print(f'\n Best NLP Model: {best_nlp_name} | Accuracy: {best_nlp_acc:.4f}')

In [ ]:
# Visualize NLP Results
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('NLP Intent Classification - Algorithm Comparison', fontsize=14, fontweight='bold')

names_nlp = list(nlp_results.keys())
accs_nlp  = [nlp_results[n]['accuracy'] for n in names_nlp]
f1s_nlp   = [nlp_results[n]['f1'] for n in names_nlp]
short_names_nlp = [n.split('. ')[1] for n in names_nlp]

colors_bar_nlp = ['gold' if a == max(accs_nlp) else 'mediumorchid' for a in accs_nlp]
axes[0].barh(short_names_nlp, accs_nlp, color=colors_bar_nlp, edgecolor='black')
axes[0].axvline(0.99, color='red', linestyle='--', label='99% target')
axes[0].set_xlim(0.9, 1.01)
axes[0].set_title('Accuracy by Algorithm'); axes[0].legend()
for i, v in enumerate(accs_nlp):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

colors_f1_nlp = ['gold' if f == max(f1s_nlp) else 'coral' for f in f1s_nlp]
axes[1].barh(short_names_nlp, f1s_nlp, color=colors_f1_nlp, edgecolor='black')
axes[1].set_xlim(0.9, 1.01)
axes[1].set_title('F1-Score by Algorithm')
for i, v in enumerate(f1s_nlp):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

cm_nlp = confusion_matrix(y_nlp_test, best_nlp_preds)
sns.heatmap(cm_nlp, annot=True, fmt='d', cmap='Purples', ax=axes[2],
            xticklabels=le_nlp.classes_, yticklabels=le_nlp.classes_)
axes[2].set_title(f'Confusion Matrix\n({best_nlp_name})')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('nlp_results.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nNLP Summary:')
print(f'   Best Model: {best_nlp_name}')
print(f'   Accuracy:   {best_nlp_acc:.4f} ({best_nlp_acc*100:.2f}%)')

---
# PART 3: SKETCH GENERATION
## Dataset Generation + 10 Algorithms + Image Rendering

### 3.1 Generate Sketch Dataset (100,000 samples with pixel/stroke features)

In [ ]:
print('Generating Sketch Generation Dataset (100,000 samples)...')

N_SKETCH = 100_000
SKETCH_CLASSES = [
    'circle', 'square', 'triangle', 'star', 'house',
    'tree', 'car', 'cat', 'dog', 'flower'
]
IMG_SIZE = 28  # 28x28 pixel sketches (like MNIST)

def generate_sketch_features(cls, n_samples):
    """Generate synthetic sketch feature vectors per class."""
    idx = SKETCH_CLASSES.index(cls)
    
    features_list = []
    for _ in range(n_samples):
        # Create a 28x28 pixel feature vector with class-specific patterns
        canvas = np.zeros((IMG_SIZE, IMG_SIZE))
        
        if cls == 'circle':
            cx, cy = IMG_SIZE//2 + np.random.randint(-3,3), IMG_SIZE//2 + np.random.randint(-3,3)
            r = IMG_SIZE//3 + np.random.randint(-3,3)
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if abs((x-cx)**2 + (y-cy)**2 - r**2) < r*1.5:
                        canvas[x,y] = np.random.uniform(0.7, 1.0)
        
        elif cls == 'square':
            s = np.random.randint(3, 7)
            cx, cy = IMG_SIZE//2 + np.random.randint(-2,2), IMG_SIZE//2 + np.random.randint(-2,2)
            canvas[cx-s:cx+s, cy-s] = np.random.uniform(0.7,1.0)
            canvas[cx-s:cx+s, cy+s] = np.random.uniform(0.7,1.0)
            canvas[cx-s, cy-s:cy+s] = np.random.uniform(0.7,1.0)
            canvas[cx+s, cy-s:cy+s] = np.random.uniform(0.7,1.0)
        
        elif cls == 'triangle':
            apex = (np.random.randint(3,8), IMG_SIZE//2 + np.random.randint(-3,3))
            bl   = (IMG_SIZE-5, apex[1]-8)
            br   = (IMG_SIZE-5, apex[1]+8)
            for i in range(IMG_SIZE):
                t = i / IMG_SIZE
                x1 = int(apex[0]*(1-t)+bl[0]*t); y1 = int(apex[1]*(1-t)+bl[1]*t)
                x2 = int(apex[0]*(1-t)+br[0]*t); y2 = int(apex[1]*(1-t)+br[1]*t)
                if 0<=x1<IMG_SIZE and 0<=y1<IMG_SIZE: canvas[x1,y1]=0.9
                if 0<=x2<IMG_SIZE and 0<=y2<IMG_SIZE: canvas[x2,y2]=0.9
        
        elif cls == 'star':
            cx, cy = IMG_SIZE//2, IMG_SIZE//2
            for i in range(10):
                angle = i * np.pi / 5
                r = IMG_SIZE//3 if i%2==0 else IMG_SIZE//6
                x = int(cx + r*np.cos(angle))
                y = int(cy + r*np.sin(angle))
                if 0<=x<IMG_SIZE and 0<=y<IMG_SIZE: canvas[x,y]=0.9
        
        elif cls == 'house':
            # Body
            canvas[14:24, 8:20] = np.random.uniform(0.6,0.9)
            canvas[14:24, 8] = 0; canvas[14:24, 19] = 0
            canvas[14, 8:20] = 0; canvas[23, 8:20] = 0
            # Roof
            for i in range(8):
                canvas[6+i, 14-i:14+i+1] = 0.8
        
        elif cls == 'tree':
            # Trunk
            canvas[18:25, 12:16] = 0.8
            # Canopy
            cx,cy = 12,14
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if (x-cx)**2+(y-cy)**2 < 64: canvas[x,y]=0.9
        
        elif cls == 'car':
            canvas[14:20, 4:24] = 0.7  # body
            canvas[10:15, 8:22] = 0.7  # cabin
            # Wheels
            for cx,cy in [(19,7),(19,21)]:
                for x in range(IMG_SIZE):
                    for y in range(IMG_SIZE):
                        if (x-cx)**2+(y-cy)**2 < 9: canvas[x,y]=0.9
        
        elif cls == 'cat':
            # Head
            cx,cy = 14,14
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if (x-cx)**2+(y-cy)**2 < 64: canvas[x,y]=0.7
            # Ears
            canvas[6:10, 8:12] = 0.8
            canvas[6:10, 16:20] = 0.8
        
        elif cls == 'dog':
            # Body
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if (x-16)**2+(y-16)**2 < 49: canvas[x,y]=0.7
            # Head
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if (x-10)**2+(y-10)**2 < 36: canvas[x,y]=0.8
            canvas[7:12, 6:10] = 0.6  # ear
        
        elif cls == 'flower':
            # Center
            for x in range(IMG_SIZE):
                for y in range(IMG_SIZE):
                    if (x-14)**2+(y-14)**2 < 16: canvas[x,y]=0.9
            # Petals
            for angle in range(0,360,60):
                rad = np.radians(angle)
                px = int(14 + 7*np.sin(rad))
                py = int(14 + 7*np.cos(rad))
                if 0<=px<IMG_SIZE and 0<=py<IMG_SIZE:
                    canvas[max(0,px-2):min(IMG_SIZE,px+3), max(0,py-2):min(IMG_SIZE,py+3)] = 0.85
        
        # Add Gaussian noise
        canvas += np.random.normal(0, 0.05, canvas.shape)
        canvas = np.clip(canvas, 0, 1)
        features_list.append(canvas.flatten())
    
    pixel_array = np.array(features_list)
    
    # Engineered features ON TOP of pixel features
    mean_brightness  = pixel_array.mean(axis=1)
    std_brightness   = pixel_array.std(axis=1)
    n_nonzero        = (pixel_array > 0.1).sum(axis=1)
    max_val          = pixel_array.max(axis=1)
    min_val          = pixel_array.min(axis=1)
    
    # Top half vs bottom half
    top_half    = pixel_array[:, :IMG_SIZE*IMG_SIZE//2].mean(axis=1)
    bottom_half = pixel_array[:, IMG_SIZE*IMG_SIZE//2:].mean(axis=1)
    left_half   = pixel_array.reshape(n_samples, IMG_SIZE, IMG_SIZE)[:, :, :IMG_SIZE//2].mean(axis=(1,2))
    right_half  = pixel_array.reshape(n_samples, IMG_SIZE, IMG_SIZE)[:, :, IMG_SIZE//2:].mean(axis=(1,2))
    center_mass = pixel_array.reshape(n_samples, IMG_SIZE, IMG_SIZE)[:, 10:18, 10:18].mean(axis=(1,2))
    
    extra = np.column_stack([mean_brightness, std_brightness, n_nonzero, max_val,
                              min_val, top_half, bottom_half, left_half, right_half, center_mass])
    
    full_features = np.hstack([pixel_array, extra])
    labels = np.array([cls] * n_samples)
    return full_features, labels

# Generate all class data
sketch_features_list = []
sketch_labels_list   = []
spc_sketch = N_SKETCH // len(SKETCH_CLASSES)

for cls in tqdm(SKETCH_CLASSES, desc='Generating sketch data'):
    feats, labs = generate_sketch_features(cls, spc_sketch)
    sketch_features_list.append(feats)
    sketch_labels_list.append(labs)

X_sketch_all = np.vstack(sketch_features_list)
y_sketch_all = np.concatenate(sketch_labels_list)

# Shuffle
perm = np.random.permutation(len(y_sketch_all))
X_sketch_all = X_sketch_all[perm]
y_sketch_all = y_sketch_all[perm]

# Encode
le_sketch = LabelEncoder()
y_sketch_enc = le_sketch.fit_transform(y_sketch_all)

print(f'\nSketch Dataset: {X_sketch_all.shape}')
print(f'Feature dimensions: {X_sketch_all.shape[1]} ({IMG_SIZE*IMG_SIZE} pixels + 10 engineered)')
print(f'Classes: {le_sketch.classes_}')

In [ ]:
# Visualize sample sketches
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Generated Sketches (28×28 pixels)', fontsize=14, fontweight='bold')

for i, cls in enumerate(SKETCH_CLASSES):
    ax = axes[i//5, i%5]
    cls_idx = np.where(y_sketch_all == cls)[0][0]
    img = X_sketch_all[cls_idx, :IMG_SIZE*IMG_SIZE].reshape(IMG_SIZE, IMG_SIZE)
    ax.imshow(img, cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(cls, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('sketch_samples.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.2 Apply 10 Algorithms for Sketch Classification

In [ ]:
# Use PCA to reduce dimensionality for some algorithms
print('Preprocessing Sketch Data (PCA for dimensionality reduction)...')

# Train-test split first
X_sk_train, X_sk_test, y_sk_train, y_sk_test = train_test_split(
    X_sketch_all, y_sketch_enc, test_size=0.2, random_state=42, stratify=y_sketch_enc
)

# Scale
scaler_sketch = StandardScaler()
X_sk_train_sc = scaler_sketch.fit_transform(X_sk_train)
X_sk_test_sc  = scaler_sketch.transform(X_sk_test)

# PCA for reduced feature space
pca_sketch = PCA(n_components=100, random_state=42)
X_sk_train_pca = pca_sketch.fit_transform(X_sk_train_sc)
X_sk_test_pca  = pca_sketch.transform(X_sk_test_sc)

print(f'Train: {X_sk_train.shape} | Test: {X_sk_test.shape}')
print(f'PCA explained variance: {pca_sketch.explained_variance_ratio_.sum():.4f}')

sketch_algorithms = {
    '1. Random Forest':      (RandomForestClassifier(n_estimators=200, max_depth=25, n_jobs=-1, random_state=42), 'raw'),
    '2. XGBoost':            (xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1,
                                                  eval_metric='mlogloss', n_jobs=-1, random_state=42), 'pca'),
    '3. LightGBM':           (lgb.LGBMClassifier(n_estimators=200, max_depth=12, n_jobs=-1,
                                                   random_state=42, verbose=-1), 'raw'),
    '4. Extra Trees':        (ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42), 'raw'),
    '5. MLP Neural Network': (MLPClassifier(hidden_layer_sizes=(512, 256, 128, 64), max_iter=500,
                                             random_state=42, early_stopping=True, batch_size=256), 'pca'),
    '6. Linear SVM':         (LinearSVC(max_iter=3000, C=1.0, random_state=42), 'pca'),
    '7. Logistic Regression':(LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, random_state=42), 'pca'),
    '8. KNN':                (KNeighborsClassifier(n_neighbors=5, n_jobs=-1), 'pca'),
    '9. Gradient Boosting':  (GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42), 'pca'),
    '10. AdaBoost':          (AdaBoostClassifier(n_estimators=150, random_state=42), 'pca'),
}

sketch_results = {}

print('\nTraining 10 Sketch Recognition Algorithms...\n')
print(f'{"Algorithm":<30} {"Accuracy":>10} {"F1-Score":>10} {"Time(s)":>10}')
print('-' * 65)

best_sketch_model = None
best_sketch_acc   = 0
best_sketch_name  = ''
best_sketch_preds = None

for name, (model, feat_type) in sketch_algorithms.items():
    start = time.time()
    
    if feat_type == 'pca':
        Xtr, Xte = X_sk_train_pca, X_sk_test_pca
    else:
        Xtr, Xte = X_sk_train, X_sk_test
    
    model.fit(Xtr, y_sk_train)
    y_pred = model.predict(Xte)
    
    acc = accuracy_score(y_sk_test, y_pred)
    f1  = f1_score(y_sk_test, y_pred, average='weighted')
    elapsed = time.time() - start
    
    sketch_results[name] = {'accuracy': acc, 'f1': f1, 'time': elapsed}
    print(f'{name:<30} {acc:>10.4f} {f1:>10.4f} {elapsed:>10.2f}s')
    
    if acc > best_sketch_acc:
        best_sketch_acc   = acc
        best_sketch_model = model
        best_sketch_name  = name
        best_sketch_preds = y_pred

print('-' * 65)
print(f'\nBest Sketch Model: {best_sketch_name} | Accuracy: {best_sketch_acc:.4f}')

In [ ]:
# Visualize Sketch Results
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('✏️ Sketch Generation/Classification - Algorithm Comparison', fontsize=14, fontweight='bold')

names_sk = list(sketch_results.keys())
accs_sk  = [sketch_results[n]['accuracy'] for n in names_sk]
f1s_sk   = [sketch_results[n]['f1'] for n in names_sk]
short_names_sk = [n.split('. ')[1] for n in names_sk]

colors_sk = ['gold' if a == max(accs_sk) else 'forestgreen' for a in accs_sk]
axes[0].barh(short_names_sk, accs_sk, color=colors_sk, edgecolor='black')
axes[0].axvline(0.99, color='red', linestyle='--', label='99% target')
axes[0].set_xlim(0.9, 1.01)
axes[0].set_title('Accuracy by Algorithm'); axes[0].legend()
for i, v in enumerate(accs_sk):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

colors_f1_sk = ['gold' if f == max(f1s_sk) else 'lightcoral' for f in f1s_sk]
axes[1].barh(short_names_sk, f1s_sk, color=colors_f1_sk, edgecolor='black')
axes[1].set_xlim(0.9, 1.01)
axes[1].set_title('F1-Score by Algorithm')
for i, v in enumerate(f1s_sk):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

cm_sk = confusion_matrix(y_sk_test, best_sketch_preds)
sns.heatmap(cm_sk, annot=True, fmt='d', cmap='Greens', ax=axes[2],
            xticklabels=le_sketch.classes_, yticklabels=le_sketch.classes_)
axes[2].set_title(f'Confusion Matrix\n({best_sketch_name})')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('sketch_results.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.3 Sketch Generation: Render Sketches from Intent

In [ ]:
def render_sketch(cls, size=128, style='clean'):
    """Render a 128x128 sketch image given a class label."""
    img = Image.new('L', (size, size), 240)
    draw = ImageDraw.Draw(img)
    margin = size // 8
    cx, cy = size//2, size//2
    
    if cls == 'circle':
        r = size//3
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline=20, width=4)
    
    elif cls == 'square':
        s = size//3
        draw.rectangle([cx-s, cy-s, cx+s, cy+s], outline=20, width=4)
    
    elif cls == 'triangle':
        pts = [(cx, margin), (margin, size-margin), (size-margin, size-margin)]
        draw.polygon(pts, outline=20)
        for i in range(len(pts)): draw.line([pts[i], pts[(i+1)%3]], fill=20, width=4)
    
    elif cls == 'star':
        pts = []
        for i in range(10):
            angle = i*np.pi/5 - np.pi/2
            r = size//3 if i%2==0 else size//6
            pts.append((cx + r*np.cos(angle), cy + r*np.sin(angle)))
        draw.polygon(pts, outline=20)
    
    elif cls == 'house':
        # Body
        draw.rectangle([margin*2, cy, size-margin*2, size-margin], outline=20, width=3)
        # Roof
        roof_pts = [(cx, margin), (margin*2-4, cy+5), (size-margin*2+4, cy+5)]
        for i in range(len(roof_pts)): draw.line([roof_pts[i], roof_pts[(i+1)%3]], fill=20, width=4)
        # Door
        draw.rectangle([cx-12, size-margin-30, cx+12, size-margin], outline=20, width=2)
        # Window
        draw.rectangle([margin*2+10, cy+15, margin*2+35, cy+40], outline=20, width=2)
    
    elif cls == 'tree':
        # Trunk
        draw.rectangle([cx-12, cy+20, cx+12, size-margin], outline=20, fill=60, width=3)
        # Canopy (3 triangles)
        for layer in range(3):
            top_y = margin + layer*25
            bot_y = cy + layer*10
            spread = size//3 + layer*10
            pts = [(cx, top_y), (cx-spread, bot_y+15), (cx+spread, bot_y+15)]
            draw.polygon(pts, fill=100, outline=20)
    
    elif cls == 'car':
        # Body
        draw.rectangle([margin, cy, size-margin, size-margin*2], outline=20, width=3, fill=150)
        # Cabin
        cabin_pts = [(margin*2, cy), (margin*2+20, cy-30), (size-margin*2-20, cy-30), (size-margin*2, cy)]
        draw.polygon(cabin_pts, outline=20, fill=170)
        # Wheels
        wr = 20
        draw.ellipse([margin+10-wr, size-margin*2-wr, margin+10+wr, size-margin*2+wr], fill=40, outline=20)
        draw.ellipse([size-margin-10-wr, size-margin*2-wr, size-margin-10+wr, size-margin*2+wr], fill=40, outline=20)
    
    elif cls == 'cat':
        # Head
        r = size//3
        draw.ellipse([cx-r, cy-r+10, cx+r, cy+r+10], outline=20, width=3, fill=180)
        # Ears
        draw.polygon([(cx-r+10, cy-r+15), (cx-r-5, cy-r-20), (cx-r+30, cy-r+5)], fill=180, outline=20)
        draw.polygon([(cx+r-10, cy-r+15), (cx+r+5, cy-r-20), (cx+r-30, cy-r+5)], fill=180, outline=20)
        # Eyes
        draw.ellipse([cx-20, cy-5, cx-8, cy+7], fill=30, outline=20)
        draw.ellipse([cx+8, cy-5, cx+20, cy+7], fill=30, outline=20)
        # Whiskers
        for dy in [-5, 0, 5]:
            draw.line([(cx-r-5, cy+dy+10), (cx-15, cy+10)], fill=30, width=2)
            draw.line([(cx+15, cy+10), (cx+r+5, cy+dy+10)], fill=30, width=2)
    
    elif cls == 'dog':
        # Body
        draw.ellipse([margin*2, cy, size-margin*2, size-margin], fill=170, outline=20, width=3)
        # Head
        r = size//4
        draw.ellipse([margin, margin*2, margin+r*2, margin*2+r*2], fill=170, outline=20, width=3)
        # Ear
        draw.ellipse([margin-10, margin*2+5, margin+15, margin*2+r], fill=120, outline=20)
        # Eye
        draw.ellipse([margin+r//2, margin*2+r//3, margin+r//2+12, margin*2+r//3+12], fill=30, outline=20)
        # Tail
        draw.arc([size-margin-20, margin*2, size-margin+20, cy], start=30, end=180, fill=20, width=4)
        # Legs
        for lx in [margin*2+15, margin*2+35, size-margin*2-35, size-margin*2-15]:
            draw.rectangle([lx-8, size-margin-25, lx+8, size-margin+15], fill=170, outline=20)
    
    elif cls == 'flower':
        # Stem
        draw.line([(cx, cy+20), (cx, size-margin)], fill=60, width=6)
        # Leaf
        draw.ellipse([cx, cy+30, cx+30, cy+60], fill=80, outline=40)
        # Petals
        r_petal = size//4
        for angle in range(0, 360, 45):
            rad = np.radians(angle)
            px = int(cx + r_petal*np.cos(rad))
            py = int(cy + r_petal*np.sin(rad))
            draw.ellipse([px-15, py-15, px+15, py+15], fill=200, outline=20)
        # Center
        draw.ellipse([cx-18, cy-18, cx+18, cy+18], fill=230, outline=20, width=3)
    
    # Add slight blur for sketch effect
    img_arr = np.array(img)
    img_arr = ndimage.gaussian_filter(img_arr, sigma=0.5)
    return Image.fromarray(img_arr.astype(np.uint8))


# Render all 10 classes
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('AI-Generated Sketches (128×128) — Full Quality', fontsize=16, fontweight='bold')
fig.patch.set_facecolor('#f8f8f8')

for i, cls in enumerate(SKETCH_CLASSES):
    ax = axes[i//5, i%5]
    sketch_img = render_sketch(cls, size=128)
    ax.imshow(sketch_img, cmap='gray')
    ax.set_title(f'{cls.upper()}', fontsize=12, fontweight='bold', pad=8)
    ax.axis('off')
    ax.set_facecolor('#f8f8f8')

plt.tight_layout(pad=2)
plt.savefig('generated_sketches.png', dpi=150, bbox_inches='tight', facecolor='#f8f8f8')
plt.show()
print('✅ Sketches rendered and saved!')

---
# PART 4: END-TO-END PIPELINE
## Speech → NLP → Sketch Generation

In [ ]:
print('Running Full End-to-End Pipeline...\n')
print('='*70)

# Test inputs
test_cases = [
    ('circle',   'draw a beautiful circle for me'),
    ('star',     'please sketch a five-pointed star'),
    ('house',    'can you draw a simple house'),
    ('cat',      'sketch a small cat please'),
    ('car',      'generate a detailed car illustration'),
]

pipeline_outputs = []

for audio_class, text_input in test_cases:
    print(f'\n[INPUT] Audio class: "{audio_class}" | Text: "{text_input}"')
    
    # --- STEP 1: Speech Recognition ---
    # Simulate: generate feature vector for the audio class
    sp_feats, _ = generate_audio_features(audio_class, 1)
    sp_feats_arr = sp_feats[speech_feature_cols].values
    
    # Use best speech model to predict
    use_sc = any(k in best_speech_name for k in ['SVM','Logistic','MLP','KNN','SGD'])
    if use_sc:
        sp_in = scaler_speech.transform(sp_feats_arr)
    else:
        sp_in = sp_feats_arr
    sp_pred = le_speech.inverse_transform(best_speech_model.predict(sp_in))[0]
    print(f'  Step 1 - Speech Recognition → "{sp_pred}"')
    
    # --- STEP 2: NLP Intent Classification ---
    processed = preprocess_text(text_input)
    nlp_vec = tfidf.transform([processed])
    dense_needed_nlp = any(k in best_nlp_name for k in ['Random Forest','Extra Trees','XGBoost',
                                                         'LightGBM','Gradient Boosting','MLP','KNN'])
    if dense_needed_nlp:
        nlp_in = nlp_vec.toarray()
    else:
        nlp_in = nlp_vec
    nlp_pred = le_nlp.inverse_transform(best_nlp_model.predict(nlp_in))[0]
    sketch_class = nlp_pred.replace('draw_', '')
    print(f'  Step 2 - NLP Intent → "{nlp_pred}" → Sketch target: "{sketch_class}"')
    
    # --- STEP 3: Sketch Generation ---
    sketch_img = render_sketch(sketch_class, size=128)
    print(f'  Step 3 - Sketch Generated for "{sketch_class}"')
    
    pipeline_outputs.append({
        'audio_input':    audio_class,
        'text_input':     text_input,
        'speech_pred':    sp_pred,
        'nlp_intent':     nlp_pred,
        'sketch_class':   sketch_class,
        'sketch_img':     sketch_img
    })

print('\n' + '='*70)
print('Pipeline complete for all test cases!')

In [ ]:
# Visualize the full pipeline outputs
fig = plt.figure(figsize=(20, 14))
fig.suptitle('End-to-End Pipeline: Speech → NLP → Sketch Generation', 
             fontsize=16, fontweight='bold', y=0.98)

n_cases = len(pipeline_outputs)

for i, out in enumerate(pipeline_outputs):
    # Sketch image
    ax_sketch = fig.add_subplot(3, n_cases, i+1)
    ax_sketch.imshow(out['sketch_img'], cmap='gray')
    ax_sketch.set_title(f'✏️ {out["sketch_class"].upper()}\n(Generated)', fontsize=11, fontweight='bold')
    ax_sketch.axis('off')
    
    # Pipeline flow text
    ax_text = fig.add_subplot(3, n_cases, n_cases+i+1)
    ax_text.axis('off')
    flow_text = (
        f'SPEECH INPUT\n"{out["audio_input"]}"\n'
        f'  ↓\n'
        f'Speech Model:\n→ "{out["speech_pred"]}"\n'
        f'  ↓\n'
        f'TEXT INPUT\n"{out["text_input"][:30]}..."\n'
        f'  ↓\n'
        f'NLP Model:\n→ "{out["nlp_intent"]}"\n'
        f'  ↓\n'
        f'SKETCH:\n"{out["sketch_class"]}"'
    )
    ax_text.text(0.5, 0.5, flow_text, ha='center', va='center',
                 fontsize=9, family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
                 transform=ax_text.transAxes)

# Overall pipeline diagram
ax_diag = fig.add_subplot(3, 1, 3)
ax_diag.axis('off')
pipeline_text = (
    'Audio Features  →  [Speech Recognition Model]  →  Word Class  →  '
    '[NLP Intent Classifier]  →  Intent  →  [Sketch Generator]  →  ✏️ Sketch Image'
)
ax_diag.text(0.5, 0.6, pipeline_text, ha='center', va='center', fontsize=13, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
             transform=ax_diag.transAxes)
ax_diag.text(0.5, 0.2,
             f'Speech Accuracy: {best_speech_acc*100:.2f}%  |  '
             f'NLP Accuracy: {best_nlp_acc*100:.2f}%  |  '
             f'Sketch Accuracy: {best_sketch_acc*100:.2f}%',
             ha='center', va='center', fontsize=12, color='darkgreen', fontweight='bold',
             transform=ax_diag.transAxes)

plt.tight_layout()
plt.savefig('full_pipeline_output.png', dpi=130, bbox_inches='tight')
plt.show()
print('Pipeline visualization complete!')

---
# 📊 PART 5: FINAL COMPREHENSIVE RESULTS SUMMARY

In [ ]:
print('\n' + '='*70)
print('FINAL RESULTS SUMMARY')
print('='*70)

models_summary = {
    'Speech Recognition': {
        'best_model': best_speech_name,
        'accuracy':   best_speech_acc,
        'dataset':    '100,000 audio feature samples',
        'algorithms': len(speech_results),
        'features':   len(speech_feature_cols)
    },
    'NLP Intent Classification': {
        'best_model': best_nlp_name,
        'accuracy':   best_nlp_acc,
        'dataset':    '100,000 text samples',
        'algorithms': len(nlp_results),
        'features':   'TF-IDF (5000 features)'
    },
    'Sketch Generation': {
        'best_model': best_sketch_name,
        'accuracy':   best_sketch_acc,
        'dataset':    '100,000 sketch pixel samples',
        'algorithms': len(sketch_results),
        'features':   f'{IMG_SIZE*IMG_SIZE} pixels + 10 engineered'
    },
}

for model_name, info in models_summary.items():
    acc_pct = info['accuracy'] * 100
    status  = 'TARGET MET' if acc_pct >= 99 else f'📈 {acc_pct:.2f}% (high accuracy)'
    print(f"\n{'─'*60}")
    print(f"  MODEL: {model_name}")
    print(f"  Best Algorithm:  {info['best_model']}")
    print(f"  Accuracy:        {acc_pct:.2f}%  {status}")
    print(f"  Dataset Size:    {info['dataset']}")
    print(f"  # Algorithms:    {info['algorithms']}")
    print(f"  Features:        {info['features']}")

print(f"\n{'='*70}")
print('END-TO-END PIPELINE STATUS: OPERATIONAL')
print(f"{'='*70}")
print('\nSaved outputs:')
for f in ['speech_dataset_analysis.png','speech_results.png',
          'nlp_dataset_analysis.png','nlp_results.png',
          'sketch_samples.png','sketch_results.png',
          'generated_sketches.png','full_pipeline_output.png']:
    print(f'   • {f}')

In [ ]:
# Final combined comparison chart
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Complete AI Pipeline — All Algorithms Across All 3 Models', 
             fontsize=14, fontweight='bold')

# Speech
names_sp_s   = [n.split('. ')[1] for n in speech_results]
accs_sp_s    = [speech_results[n]['accuracy'] for n in speech_results]
c_sp = ['gold' if a == max(accs_sp_s) else '#4A90E2' for a in accs_sp_s]
axes[0].barh(names_sp_s, accs_sp_s, color=c_sp, edgecolor='black')
axes[0].axvline(0.99, color='red', linestyle='--', linewidth=2, label='99% target')
axes[0].set_xlim(0.9, 1.01)
axes[0].set_title('🎙️ Speech Recognition', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Accuracy'); axes[0].legend()

# NLP
names_nlp_s = [n.split('. ')[1] for n in nlp_results]
accs_nlp_s  = [nlp_results[n]['accuracy'] for n in nlp_results]
c_nlp = ['gold' if a == max(accs_nlp_s) else '#9B59B6' for a in accs_nlp_s]
axes[1].barh(names_nlp_s, accs_nlp_s, color=c_nlp, edgecolor='black')
axes[1].axvline(0.99, color='red', linestyle='--', linewidth=2, label='99% target')
axes[1].set_xlim(0.9, 1.01)
axes[1].set_title('NLP Intent Classification', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].legend()

# Sketch
names_sk_s  = [n.split('. ')[1] for n in sketch_results]
accs_sk_s   = [sketch_results[n]['accuracy'] for n in sketch_results]
c_sk = ['gold' if a == max(accs_sk_s) else '#27AE60' for a in accs_sk_s]
axes[2].barh(names_sk_s, accs_sk_s, color=c_sk, edgecolor='black')
axes[2].axvline(0.99, color='red', linestyle='--', linewidth=2, label='99% target')
axes[2].set_xlim(0.9, 1.01)
axes[2].set_title('Sketch Classification', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Accuracy'); axes[2].legend()

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('\nALL DONE! Full AI Pipeline with 100K+ samples & 35 total algorithms complete!')

# THE END